In [ ]:
import sys
from pathlib import Path

# =========================================================
# 1. Make the project root available to Python
# =========================================================

# This notebook is stored in:
# Enterprise-Banking-Analytics-Platform/notebooks/
#
# Therefore, its parent directory is the project root.
PROJECT_ROOT_PATH = Path.cwd().parent

if str(PROJECT_ROOT_PATH) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT_PATH))


# =========================================================
# 2. Import project configuration and reusable pipeline code
# =========================================================

from configs.config import (
    BRONZE_PATH,
    LANDING_PATH,
    METADATA_PATH,
)

from configs.source_schemas import (
    TRANSACTION_EXPECTED_DTYPES,
    TRANSACTION_REQUIRED_COLUMNS,
)

from src.bronze_ingestion import ingest_csv_to_bronze
from src.metadata_control import load_control_table


# =========================================================
# 3. Define source, Bronze, and metadata paths
# =========================================================

source_file = (
    LANDING_PATH
    / "User0_credit_card_transactions.csv"
)

bronze_output_path = (
    BRONZE_PATH
    / "credit_card_transactions_sample.parquet"
)

control_table_path = (
    METADATA_PATH
    / "file_processing_control.parquet"
)


# =========================================================
# 4. Display the pipeline inputs
# =========================================================

print("Source file:", source_file)
print("Source file exists:", source_file.exists())
print("Bronze output:", bronze_output_path)
print("Control table:", control_table_path)


# =========================================================
# 5. Run the Bronze ingestion pipeline
# =========================================================

result = ingest_csv_to_bronze(
    source_file=source_file,
    bronze_output_path=bronze_output_path,
    control_table_path=control_table_path,
    expected_columns=TRANSACTION_REQUIRED_COLUMNS,
    expected_dtypes=TRANSACTION_EXPECTED_DTYPES,
)


# =========================================================
# 6. Display the pipeline result
# =========================================================

print("\nPipeline result:")

for key, value in result.items():
    print(f"{key}: {value}")


# =========================================================
# 7. Verify the Bronze output when it exists
# =========================================================

if bronze_output_path.exists():
    import pandas as pd

    bronze_check_df = pd.read_parquet(bronze_output_path)

    required_audit_columns = [
        "_source_file_name",
        "_ingestion_timestamp_utc",
        "_pipeline_run_id",
    ]

    print("\nBronze verification:")
    print("Bronze file exists:", True)
    print("Bronze rows:", len(bronze_check_df))
    print("Bronze columns:", len(bronze_check_df.columns))

    for column in required_audit_columns:
        print(
            f"{column}:",
            column in bronze_check_df.columns,
        )

    assert all(
        column in bronze_check_df.columns
        for column in required_audit_columns
    )

    assert bronze_check_df[
        "_source_file_name"
    ].notna().all()

    assert bronze_check_df[
        "_ingestion_timestamp_utc"
    ].notna().all()

    assert bronze_check_df[
        "_pipeline_run_id"
    ].notna().all()

    print("Bronze verification passed.")


# =========================================================
# 8. Display the metadata/control table
# =========================================================

control_check_df = load_control_table(
    control_table_path
)

print("\nControl table records:")
display(control_check_df)



